Analysing the movie dataset from TMDB (Downloaded on August 2024)

In [ ]:
import pandas

movies=pandas.read_csv('../TMDB_movie_dataset_v11.csv')
movies.head(3)

In [ ]:
movies.describe()

In [ ]:
movies.info()

In [ ]:
movies.isnull().sum()

In [ ]:
movies.columns

Taking the relavant data into a new dataframe, deleting empty data, deleting movies with less than 200 votes

In [ ]:
mv1=movies[['id', 'title', 'vote_average', 'vote_count', 'overview', 'genres', 'keywords']]

mv1=mv1.dropna(how="any",axis=0)

mv1 = mv1[mv1['vote_count'] >= 200]

#mv1.isnull().sum()
#mv1.head(3)

mv1['tags'] = mv1['overview'] + mv1['genres'] + mv1['keywords']
mv1 = mv1.drop(columns=['vote_count', 'overview', 'keywords'])
print(mv1[mv1['title'] == "The Dark Knight"]['tags'].values[0])
mv1

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
datasize = mv1.shape[0]
cv=CountVectorizer(max_features=datasize, stop_words='english')
cv


In [ ]:
vector1=cv.fit_transform(mv1['tags'].values.astype('U')).toarray()
vector1.shape

In [ ]:

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

# vector 1 is very large and direct implementation of cosine similarity command is very resource intensive
# Ensuring it's in sparse format
vector1_sparse = csr_matrix(vector1)

# Defining the chunk size for processing
chunk_size = 1000

# Initializing an empty array to store the similarity results
similarity_matrix = np.zeros((vector1_sparse.shape[0], vector1_sparse.shape[0]))

# Computing cosine similarity in chunks
for i in range(0, vector1_sparse.shape[0], chunk_size):
    end_index = min(i + chunk_size, vector1_sparse.shape[0])
    similarity_matrix[i:end_index] = cosine_similarity(vector1_sparse[i:end_index], vector1_sparse)


In [ ]:
similarity_matrix, similarity_matrix.shape

In [ ]:
def recommendme(movie):
    index=mv1[mv1['title']==movie].index[0]

    distance = sorted(list(enumerate(similarity_matrix[index])), reverse=True, key=lambda vector1:vector1[1])

    for i in distance[0:5]:
        print(mv1.iloc[i[0]].title)

In [ ]:
recommendme("The Sixth Sense")

In [ ]:
import pickle

pickle.dump(mv1, open('movies.pkl', 'wb'))

pickle.dump(similarity_matrix, open('similarity.pkl', 'wb'))

In [ ]:
pickle.load(open('movies.pkl', 'rb'))